# MyCON Universal HQ Reconstruction

Choose the source and quality below, then use **Runtime → Run all**. The default is **Auto HQ**. Restart with the same inputs and profile to restore SHA256-verified checkpoints. On Colab, use a GPU runtime for the optional COLMAP PatchMatch backend. The CPU route stays available.

You receive point-cloud geometry, camera calibration and poses, source-view comparisons and an offline viewer. Open `Quality_Report.json` to see observed coverage and the selected geometry backend. A disconnected or degenerate clip can stop or yield partial geometry. Point-cloud coordinates have an arbitrary scale unless independently controlled.

In [ ]:
# @title 1. Input and quality
INPUT_MODE = "Drive" # @param ["Drive", "Upload"]
INPUT_PATHS = "" # @param {type:"string"}
QUALITY_PROFILE = "Auto HQ" # @param ["Auto HQ", "Fast Preview", "High Quality", "Ultra Quality", "Long Video", "Low VRAM", "Scientific Benchmark"]
SAVE_RESUME_TO_DRIVE = True # @param {type:"boolean"}

# A Drive file or directory. Separate several paths with a vertical bar.
# Example: MyDrive/MyCON/site-a.mp4 | MyDrive/MyCON/site-b.mov
if INPUT_MODE == "Drive" and not INPUT_PATHS.strip():
    INPUT_PATHS = input("Drive file or folder. Separate multiple paths with |: ").strip()

In [ ]:
# @title 2. Environment and pinned engine
import os
from pathlib import Path
import shutil
import subprocess
import sys

CODE_REF = "1235af9a3063c80706ac30729a1e0c425d118755"
BASE = Path("/content/mycon-universal")
CHECKOUT = BASE / "source"
BASE.mkdir(parents=True, exist_ok=True)
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Select a Colab runtime with Python 3.11 or 3.12 for this pinned wheel set.")
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if not (CHECKOUT / ".git").is_dir():
    CHECKOUT.mkdir(exist_ok=True)
    subprocess.run(["git", "init", "-q", str(CHECKOUT)], check=True)
    subprocess.run(["git", "-C", str(CHECKOUT), "remote", "add", "origin",
                    "https://github.com/sayyedalimrj/MyCon.git"], check=True)
exists = subprocess.run(["git", "-C", str(CHECKOUT), "cat-file", "-e", CODE_REF + "^{commit}"],
                        capture_output=True).returncode == 0
if not exists:
    subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth=1", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "--quiet", CODE_REF], check=True)
PROJECT = CHECKOUT / "repository"
GPU = False
if shutil.which("nvidia-smi"):
    hw = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                         "--format=csv,noheader"], text=True, capture_output=True, timeout=8)
    GPU = hw.returncode == 0 and bool(hw.stdout.strip())
    if GPU:
        print(hw.stdout.strip())
ENV_DIR = BASE / ("venv-cuda12" if GPU else "venv-cpu")
if not (ENV_DIR / "bin/python").is_file():
    subprocess.run([sys.executable, "-m", "venv", str(ENV_DIR)], check=True)
PY = str(ENV_DIR / "bin/python")
WHEEL = "pycolmap-cuda12==4.2.1" if GPU else "pycolmap==4.2.1"
try:
    subprocess.run([PY, "-m", "pip", "--disable-pip-version-check", "install", "-q",
                    "-r", str(PROJECT / "requirements-universal-hq.txt"), WHEEL], check=True)
    subprocess.run([PY, "-c", "import pycolmap, cv2; print('COLMAP CUDA compiled:', pycolmap.has_cuda)"],
                   check=True)
except subprocess.CalledProcessError:
    if not GPU:
        raise
    print("CUDA wheel unavailable in this runtime. Switching to calibrated CPU stereo.")
    CPU_ENV = BASE / "venv-cpu"
    subprocess.run([sys.executable, "-m", "venv", str(CPU_ENV)], check=True)
    PY = str(CPU_ENV / "bin/python")
    subprocess.run([PY, "-m", "pip", "--disable-pip-version-check", "install", "-q", "-r",
                    str(PROJECT / "requirements-universal-hq.txt"), "pycolmap==4.2.1"], check=True)
print("Local free disk GiB:", round(shutil.disk_usage(BASE).free / 2**30, 1))

In [ ]:
# @title 3. Prepare the job
from google.colab import drive, files
from hashlib import sha256
import json

DRIVE = Path("/content/drive")
if INPUT_MODE == "Drive" or SAVE_RESUME_TO_DRIVE:
    drive.mount(str(DRIVE))
if INPUT_MODE == "Drive":
    def drive_path(value):
        path = Path(value.strip()).expanduser()
        if path.is_absolute():
            return path
        if path.parts and path.parts[0] in {"MyDrive", "Shareddrives"}:
            return DRIVE / path
        return DRIVE / "MyDrive" / path
    selected = [drive_path(path) for path in INPUT_PATHS.split("|") if path.strip()]
else:
    upload_dir = BASE / "uploads"
    upload_dir.mkdir(exist_ok=True)
    origin = Path.cwd()
    try:
        os.chdir(upload_dir)
        selected = [upload_dir / Path(name).name for name in files.upload()]
    finally:
        os.chdir(origin)
if not selected or any(not item.exists() for item in selected):
    raise FileNotFoundError("Select an existing media file or image folder.")
env = dict(os.environ, PYTHONPATH=str(PROJECT))
identity_script = """from pathlib import Path
import json,sys
from reconstruction.universal.inputs import list_sources
from reconstruction.workflow import sha256
items=list_sources([Path(arg) for arg in sys.argv[1:]])
print(json.dumps([(x.name, sha256(x)) for x in items]))
"""
fingerprints = json.loads(subprocess.check_output(
    [PY, "-c", identity_script, *map(str, selected)], env=env, text=True))
run_token = sha256(json.dumps([CODE_REF, QUALITY_PROFILE, fingerprints]).encode()).hexdigest()[:20]
WORKSPACE = BASE / "runs" / run_token
WORKSPACE.mkdir(parents=True, exist_ok=True)
MIRROR = DRIVE / "MyDrive" / "MyCON_Universal_HQ" / run_token if SAVE_RESUME_TO_DRIVE else None
if MIRROR:
    MIRROR.mkdir(parents=True, exist_ok=True)
print(f"Run {run_token} · {len(fingerprints)} media item(s)")

In [ ]:
# @title 4. Reconstruct or resume
import datetime

command = [PY, "-u", "-m", "reconstruction.universal.runner", *map(str, selected),
           "--out", str(WORKSPACE), "--profile", QUALITY_PROFILE]
if MIRROR:
    command += ["--checkpoint-mirror", str(MIRROR)]
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
log_path = (MIRROR or WORKSPACE) / f"execution-{stamp}.log"
with log_path.open("a", buffering=1) as log:
    with subprocess.Popen(command, cwd=PROJECT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                log.write(line)
                if line.startswith(("intake ", "features ", "matches ", "map ", "undistort ",
                                    "dense ", "export ", "Run stopped", "{")):
                    print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        code = process.wait()
if code:
    raise RuntimeError(f"Reconstruction stopped at an incomplete stage. See {log_path}")

In [ ]:
# @title 5. Package and get the model
import zipfile
from IPython.display import display, FileLink

export_dir = WORKSPACE / "export"
report = json.loads((export_dir / "Quality_Report.json").read_text())
print(f"Geometry status: {report['status']}")
print("Includes metric scale:", report["metric_validated"])
print("Full resolution PLY:", report["full_geometry"])
archive = (MIRROR or WORKSPACE) / f"MyCON_{run_token}_{stamp}.zip"
extras = [WORKSPACE / "intake/capture_analysis.json", WORKSPACE / "run_identity.json",
          WORKSPACE / "progress.jsonl", log_path]
records = {}
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    for item in [*sorted(export_dir.rglob("*")), *extras]:
        if not item.is_file():
            continue
        label = item.relative_to(export_dir).as_posix() if item.is_relative_to(export_dir) else item.name
        hasher = sha256()
        with item.open("rb") as src, z.open(label, "w", force_zip64=True) as dst:
            for block in iter(lambda: src.read(1024 * 1024), b""):
                hasher.update(block)
                dst.write(block)
        records[label] = {"sha256": hasher.hexdigest(), "bytes": item.stat().st_size}
    z.writestr("SHA256_manifest.json", json.dumps(records, indent=2))
print(archive)
if not MIRROR:
    files.download(str(archive))
else:
    display(FileLink(str(archive)))
print("Extract the entire ZIP. Open Viewer.html next to frames/ and tiles/ with a WebGL2 browser.")